# **ÚLTIMA VERSIÓN -> Martes 26 de mayo 23:22**


#**Dataset agencias de PAMI**

Se busca identificar desde el dataset obtenido en Datos.gob.ar
- Agencias de pami

---

**NOTAS DE IMPORTANCIA SOBRE EL DATASET**
Mediante una busqueda se va ofrecer las dos opciones cercanas al usuario, bajo la idea de que pueda acercarse a alguna de ellas a realizar el tramite correspondiente.


In [ ]:
# 1. IMPORTAR LIBRERÍAS
# =========================

import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

## **1. Carga de la base de agencias de PAMI**

**Opción 1**
Usar la URL del dataset, permite tener la base actualizada automaticamente

http://datos.pami.org.ar/dataset/6fda9ef9-7dab-4a1a-879d-a963f05c7fde/resource/f6e33659-84c9-4e97-a271-f9023c8c891c/download/listado-de-agencias-.xlsx

**Opción 2**
En caso de que no corra del URL, lo que puede deberse a que la página no fluye (problema que sucedio en google colab) se sube el dataset descargado de dicho link

---
**NOTA**

El codigo va a ejecutar la opcion 1, pero si la web de pami no responde,  entonces va a ejecutar la opción 2, que basicamente es un archivo de respaldo dejado en el codigo.
Este archivo de respaldo se sube al codigo una vez que se descargo desde el link que google colab no pudo ejecutar, eso implica que trae exactamente el mismo data (columnas) pero sin asegurar que esta actualizado en tiempo real. Motivo por el cual se deja el mensaje donde esto se aclare

In [ ]:
# OPCIÓN 1 + y OPCION 2 (respaldo): CARGA DEL DATASET PAMI
# ==================================================

import pandas as pd

# 1. URL oficial del dataset de agencias de PAMI
url_agencias = "http://datos.pami.org.ar/dataset/6fda9ef9-7dab-4a1a-879d-a963f05c7fde/resource/f6e33659-84c9-4e97-a271-f9023c8c891c/download/listado-de-agencias-.xlsx"

# 2. Archivo de respaldo guardado
  # En Colab, por ahora, este archivo debe estar subido a la sesión o montado desde Drive.
archivo_agencias_respaldo = "/content/drive/MyDrive/Licenciatura en ciencia política/En curso/Ciencia de datos para polítologos/Proyecto final/CODIGO_PROYECTO/CODIGO_AGENCIAS/Dataset_Agencias_PAMI.xlsx"

try:
# 3. Intentamos cargar la base desde internet
    df_agencias = pd.read_excel(url_agencias)
    print("La base se cargó correctamente desde PAMI.\n")

# 4. Si falla la conexión, usamos la copia de respaldo del proyecto
except Exception as error:
    print("No se pudo cargar la base desde PAMI en este momento. \n Se usará la última versión de respaldo disponible en el proyecto, puede la ubicación de algunas agencias no esten actualizadas. \n")

    df_agencias = pd.read_excel(archivo_agencias_respaldo)
    print("La base se cargó correctamente desde el archivo de respaldo. \n")

# 5. Mostramos las primeras filas
df_agencias.head()

#** 2. Arreglar el encabezado de la tabla**
Implica tomar la fila correcta y eliminar las innecesarias, haciendo una copia para no romperla porque si se sigue ejecutando se van a eliminar filas contantemente

In [ ]:
# ESTO SOLO SE TIENE QUE EJECUTAR UNA SOLA VEZ
# 1. Se crea una copia para limpiar sin modificar el original
df_agencias_arreglado = df_agencias.copy()

# 2. Se toma la fila 1 como encabezado real
df_agencias_arreglado.columns = df_agencias_arreglado.iloc[1]

# 3. Eliminar filas basura
df_agencias_arreglado = df_agencias_arreglado.iloc[2:]

# 3. Reiniciar índice
df_agencias_arreglado = df_agencias_arreglado.reset_index(drop=True)

# 4. Mostrar resultado
df_agencias_arreglado.head()

## **3. Exploración del dataset arreglado**

1. Nombres de filas
2. Cantidad de filas y columnas
3. Revisar la falta de datos
4. Ver el tipo de datos, si es texto o numero

In [ ]:
# 1. Ver nombres de columnas
df_agencias_arreglado.columns

In [ ]:
# 2. Cantidad de filas y columnas
df_agencias_arreglado.shape

In [ ]:
# 3. Ver si hay valores faltantes por columna

print ("Como las columnas de coordenadas son las unicas que tienen faltantes no es un problema real porque no son columnas a utilizar")

df_agencias_arreglado.isnull().sum()

In [ ]:
# 4. Tipos de datos de las columnas
df_agencias_arreglado.info()

## **4. Limpieza básica del dataset arreglado**

La limpieza va a permitir eliminar espacios en blanco al inicio y al final de las celdas y evitar errores por diferencias de formato, como escribir en mayusculas o minusculas.

In [ ]:
# 1. Crear copia del dataset original para tener un respaldo
df_agencias_limpio = df_agencias_arreglado.copy()

# 2. Eliminar columnas que no son de utilidad para el proyecto
  # Se elimina c_agencia ID pero se dejan por ahora X ID y Y ID po si posteriormente se quieren usar las coordenadas
df_agencias_limpio = df_agencias_limpio.drop(columns=["c_agencia ID"])

# 3. Se renombran las columnas para que sean más claras y fáciles de entender
df_agencias_limpio = df_agencias_limpio.rename(columns={"c_ugl ID": "ID_UGL",
                                          "d_ugl ID": "Ubicacion_Territorial",
                                          "d_agencia ID": "Nombre_Agencia",
                                          "localidad_desc": "Localidad"})

# 4. Limpiar nombres de columnas para que queden en mayuscula
df_agencias_limpio.columns = (df_agencias_limpio.columns
                                  .str.strip()
                                  .str.upper())

# 5. Limpiar espacios en columnas de texto para eliminar espacion en blanco al principio y al final
for columna in df_agencias_limpio.select_dtypes(include="object"):
    df_agencias_limpio[columna] = (df_agencias_limpio[columna]
                                       .astype(str)
                                       .str.strip())

# 6. Mostrar columnas luego de la limpieza
df_agencias_limpio.columns

## **5. Visualización de UBICACIÓN_TERRITORIAL**

Lo pongo para que se vea la clasificación que se da segun la primer columna "c_ugl ID"

In [ ]:
# Tomo el data ya limpio y me fijo qué UBICACION_TERRITORIAL corresponde a cada ID_UGL
  # Y las sumo para ver cuantas hay por ubicación

tabla_ugl = (df_agencias_limpio.groupby(["ID_UGL", "UBICACION_TERRITORIAL"]).size().reset_index(name="CANTIDAD_AGENCIAS").sort_values("ID_UGL"))

display(tabla_ugl)

# **6. Agrupamiento por provincia**

El fragmento de codigo anterior muestra que Buenos Aires, Córdoba y Santa Fe no tienen agrupadas sus agencias, asi que se agrupan para que el usuario pueda posteriormente seleccionar su provincia y reducir los datos que se le brindan

In [ ]:
# 1. Se copia la información actual de UBICACION_TERRITORIAL porque la mayoría de las provincias ya están correctamente clasificadas

df_agencias_limpio["PROVINCIA"] = df_agencias_limpio["UBICACION_TERRITORIAL"]


# 2. Reagrupar manualmente las UGL que pertenecen a Provincia de Buenos Aires
  # Incluyendo CABA porque PAMI las separa administrativamente
ugl_buenos_aires = ["BAHIA BLANCA",
                    "CAPITAL FEDERAL",
                    "LA PLATA",
                    "SAN MARTIN",
                    "LANUS",
                    "MAR DEL PLATA",
                    "MORON",
                    "AZUL",
                    "JUNIN",
                    "LUJAN",
                    "SAN JUSTO",
                    "QUILMES",
                    "CHIVILCOY"]

df_agencias_limpio.loc[df_agencias_limpio["UBICACION_TERRITORIAL"].isin(ugl_buenos_aires),"PROVINCIA"] = "BUENOS AIRES"

# 3. Reagrupar Rio Cuarto dentro de Córdoba
df_agencias_limpio.loc[df_agencias_limpio["UBICACION_TERRITORIAL"] == "RIO CUARTO","PROVINCIA"] = "CORDOBA"

# 4. Reagrupar Concordia dentro de Entre Ríos
df_agencias_limpio.loc[df_agencias_limpio["UBICACION_TERRITORIAL"] == "CONCORDIA","PROVINCIA"] = "ENTRE RIOS"


# 5. Revisar cómo quedó la agrupación

tabla_provincias = (
    df_agencias_limpio
    .groupby(["PROVINCIA", "UBICACION_TERRITORIAL"])
    .size()
    .reset_index(name="CANTIDAD_AGENCIAS")
    .sort_values(["PROVINCIA", "UBICACION_TERRITORIAL"]))

display(tabla_provincias)

## **7. Búsqueda de agencias PAMI por provincia y localidad**

En esta parte se le pide al usuario provincia y de su ubicación más cercana (localidad)

In [ ]:
# Ver provincias disponibles para el usuario

provincias_disponibles = sorted(df_agencias_limpio["Provincia"].dropna().unique())

for i, provincia in enumerate(provincias_disponibles, start=1):
    print(i, "-", provincia)

In [ ]:
# 1. Se ordenan las provincias alfabeticamente
provincias_disponibles = sorted(
    df_agencias_limpio["Provincia"].dropna().unique())

# 2. Mostrar provincias numeradas para que el usuario pueda seleccionar una
print("Seleccione su provincia:\n")
for i, provincia in enumerate(provincias_disponibles, start=1):
    print(i, "-", provincia)

# 3. Pedir selección al usuario
seleccion_provincia = input("\nIngrese el número de su provincia: ").strip()

# 4. Verificar que el usuario escribió un número
if seleccion_provincia.isdigit():
    seleccion_provincia = int(seleccion_provincia)

    # 5. Revisar que el número exista dentro de las opciones disponibles
    if 1 <= seleccion_provincia <= len(provincias_disponibles):
        provincia_elegida = provincias_disponibles[seleccion_provincia - 1]
        # En tal caso se guarda en una nueva variable -> provincia elegida
        print(f"\nProvincia seleccionada: {provincia_elegida}")
    else:
        print("\nEl número ingresado no corresponde a ninguna provincia.")
else:
    print("\nDebe ingresar solamente números.")

In [ ]:
# 1. Filtrar agencias de la provincia seleccionada
agencias_provincia = df_agencias_limpio[df_agencias_limpio["Provincia"] == provincia_elegida]

# 2. Obtener ubicaciones territoriales disponibles dentro de esa provincia
ubicaciones_disponibles = sorted(
    agencias_provincia["Ubicacion_Territorial"]
    .dropna()
    .unique())

# 3. Mostrar ubicaciones numeradas
print(f"Ubicaciones disponibles en {provincia_elegida}:\n")

for i, ubicacion in enumerate(ubicaciones_disponibles, start=1):
    print(i, "-", ubicacion)

# 4. Pedir selección al usuario
seleccion_ubicacion = input("\nIngrese el número de la ubicación más cercana: ").strip()

# 5. Validar selección
if seleccion_ubicacion.isdigit():
    seleccion_ubicacion = int(seleccion_ubicacion)

    if 1 <= seleccion_ubicacion <= len(ubicaciones_disponibles):
        ubicacion_elegida = ubicaciones_disponibles[seleccion_ubicacion - 1]
        print(f"\nUbicación seleccionada: {ubicacion_elegida}")

    else:
        print("\nEl número ingresado no corresponde a ninguna ubicación.")

else:
    print("\nDebe ingresar solamente números.")

# **8. Selección de agencias**
Al finalizar esta sección el usuario va a contar con 2 agencias selecionadas y cercanas a su ubicación, las mismas se van a guardar con dirección en el PDF y se van a presentar al usuario para que las confirme

In [ ]:
# 1. FILTRAR LAS AGENCIAS SEGÚN LO QUE EL USUARIO ELIGIÓ
# =========================================================

# Se toman únicamente las agencias que pertenecen a la provincia y a la ubicación territorial elegidas anteriormente
agencias_ubicacion = df_agencias_limpio[
    (df_agencias_limpio["Provincia"] == provincia_elegida) &
    (df_agencias_limpio["Ubicacion_Territorial"] == ubicacion_elegida)]

# Se buscan las localidades existentes dentro de esa ubicación territorial
localidades_disponibles = sorted(agencias_ubicacion["Localidad"] # Ordena alfabéticamente
    .dropna() # .Elimina valores vacíos
    .unique()) # Evita localidades repetidas


# 3. MOSTRAR LOCALIDADES NUMERADAS
# =========================================================

print(f"Localidades disponibles en {ubicacion_elegida}:\n")

# Con enumerate se da numero automaticamente a cada localidad y con start=1 hace que la numeración comience desde 1 y no desde 0
for i, localidad in enumerate(localidades_disponibles, start=1):
    print(i, "-", localidad)

# 4. Pedir al usuario que elija la localidad
  # Eliminando espacion en blanco con strip
seleccion_localidad = input("\nIngrese el número de su localidad o la más cercana: ").strip()

# 5. Validar que el dato ingresado sea un numero que exista
if seleccion_localidad.isdigit():
    # Se convierte el texto ingresado a número entero
    seleccion_localidad = int(seleccion_localidad)

    # Se revisa que el número exista dentro de la lista
    if 1 <= seleccion_localidad <= len(localidades_disponibles):

        # Se obtiene la localidad elegida y Se resta 1 porque las listas empiezan desde 0
        localidad_elegida = localidades_disponibles[seleccion_localidad - 1]

        print(f"\nLocalidad seleccionada: {localidad_elegida}")

        # 6. Filtrar agencias de la localidad seleccionada
        agencias_localidad = agencias_ubicacion[agencias_ubicacion["Localidad"] == localidad_elegida]

        # 7. Reinicir indice para que sea ordenado y mostrar las agencias disponibles
        opciones_agencias = agencias_localidad.reset_index(drop=True)
        opciones_agencias.index = opciones_agencias.index + 1
        print("\nAgencias disponibles:\n")

        display(opciones_agencias[["Nombre_Agencia", "domicilio ID", "Localidad"]])

        # 9. Crear una lista donde se guardan las agencias seleccionadas
        agencias_seleccionadas = []

        # 10. Contar la cantidad de agencias en esa localidad
        cantidad_agencias = len(opciones_agencias)

          # Si hay dos o más agencias el usuario puede elegir ambas dentro de la misma localidad
        if cantidad_agencias >= 2:
            seleccion_agencia_1 = input("\nIngrese el número de la primera agencia seleccionada: ").strip()
            seleccion_agencia_2 = input("\nIngrese el número de la segunda agencia seleccionada: ").strip()

            # Validar los datos ingresados sean numeros
            if (seleccion_agencia_1.isdigit() and seleccion_agencia_2.isdigit()):
                seleccion_agencia_1 = int(seleccion_agencia_1)
                seleccion_agencia_2 = int(seleccion_agencia_2)

                # Validar entonces que esos numeros existan
                if (seleccion_agencia_1 in opciones_agencias.index and seleccion_agencia_2 in opciones_agencias.index):

                    # Se guardan las filas seleccionadas
                    agencia_1 = opciones_agencias.loc[seleccion_agencia_1]
                    agencia_2 = opciones_agencias.loc[seleccion_agencia_2]


                    # Guardar las agencias en la lista
                    agencias_seleccionadas.append(agencia_1)
                    agencias_seleccionadas.append(agencia_2)

                    # Convertir la lista en dataframe para presentarla
                    df_agencias_seleccionadas = pd.DataFrame(agencias_seleccionadas)

                    # Mostrariamos el resultado final si ya se seleccionaron 2 agencias
                    print("\nAgencias seleccionadas:\n")

                    display(df_agencias_seleccionadas[["Nombre_Agencia","domicilio ID","Localidad"]])

                else:
                    print("\nUno de los números ingresados no corresponde a una agencia disponible.")

            else:
                print("\nDebe ingresar solamente números.")


        # 12. Si solo hay una agencia en la localidad seleccionada se guarda automáticamente esa agencia y luego se pide una segunda localidad cercana
        elif cantidad_agencias == 1:
            agencia_1 = opciones_agencias.loc[1]
            agencias_seleccionadas.append(agencia_1)

            print("\nEsta localidad tiene una sola agencia disponible."" Se guardará esa agencia y se solicitará una segunda localidad cercana.")
            print("\nSeleccione una segunda localidad cercana:\n")


            # Se muestran de neuvo las localidades disponibles
            for i, localidad in enumerate(localidades_disponibles,start=1):
                print(i, "-", localidad)

            # Se pide una segunda localidad y se validad el dato ingresado
            seleccion_localidad_2 = input("\nIngrese el número de una segunda localidad cercana: ").strip()

            if seleccion_localidad_2.isdigit():
                seleccion_localidad_2 = int(seleccion_localidad_2)

                if (1 <= seleccion_localidad_2 <=
                    len(localidades_disponibles)):

                    localidad_elegida_2 = localidades_disponibles[seleccion_localidad_2 - 1]

                    print(f"\nSegunda localidad seleccionada: {localidad_elegida_2}")


                    # Se filtran las agencias de la segunda localidad
                    agencias_localidad_2 = agencias_ubicacion[agencias_ubicacion["Localidad"] == localidad_elegida_2]

                    # Reiniciar indices
                    opciones_agencias_2 = (agencias_localidad_2.reset_index(drop=True))
                    opciones_agencias_2.index = (opciones_agencias_2.index + 1)

                    # Mostrar agencias disponibles de esa localidad
                    print("\nAgencias disponibles en la segunda localidad:\n")

                    display(opciones_agencias_2[["Nombre_Agencia","domicilio ID","Localidad"]])

                    # Contar la cantidad de agencias en la segunda localidad
                    cantidad_agencias_2 = len(opciones_agencias_2)

                    # Si la segunda localidad tiene una sola agencia, se guarda automáticamente
                    if cantidad_agencias_2 == 1:
                        agencia_2 = opciones_agencias_2.loc[1]

                        # Gurda la segunda agencia y se convierte la lista final en dataframe
                        agencias_seleccionadas.append(agencia_2)

                        df_agencias_seleccionadas = pd.DataFrame(agencias_seleccionadas)

                        print("\nLa segunda localidad tiene una sola agencia disponible. Se guardará automáticamente.")

                        # Se muestra el resultado final
                        print("\nAgencias seleccionadas:\n")

                        display(df_agencias_seleccionadas[["Nombre_Agencia","domicilio ID","Localidad"]])

                    # Si la segunda localidad tiene más de una agencia, se pide al usuario la segunda agencia
                    elif cantidad_agencias_2 >= 2:

                        # Se pide al usuario la segunda agencia y se valida el dato ingresado
                        seleccion_agencia_2 = input("\nIngrese el número de la segunda agencia seleccionada: ").strip()

                        if seleccion_agencia_2.isdigit():
                            seleccion_agencia_2 = int(seleccion_agencia_2)

                            if (seleccion_agencia_2 in opciones_agencias_2.index):
                                agencia_2 = opciones_agencias_2.loc[seleccion_agencia_2]

                                # Gurda la segunda agencia y se convierte la lista final en dataframe
                                agencias_seleccionadas.append(agencia_2)

                                df_agencias_seleccionadas = pd.DataFrame(agencias_seleccionadas)

                                # Se muestra el resultado final
                                print("\nAgencias seleccionadas:\n")

                                display(df_agencias_seleccionadas[["Nombre_Agencia","domicilio ID","Localidad"]])

                            else:
                                print("\nEl número ingresado no corresponde a ninguna agencia disponible.")

                        else:
                            print("\nDebe ingresar solamente números.")

                    else:
                        print("\nNo hay agencias disponibles en la segunda localidad seleccionada.")

                else:
                    print("\nEl número ingresado no corresponde a ninguna localidad.")

            else:
                print("\nDebe ingresar solamente números.")

        else:
            print("\nNo hay agencias disponibles para la localidad seleccionada.")

    else:
        print("\nEl número ingresado no corresponde a ninguna localidad.")

else:
    print("\nDebe ingresar solamente números.")